Run one Qwen condition at a time. Use a GPU runtime, and add Colab secrets named `GITHUB_TOKEN` and `HF_TOKEN`.

Set `BRANCH` below, run the setup cells, then pick a condition. Change the dropdown and run the last cell again for the next one. `before_question` is the before-the-question condition.

Finished problems are appended to `MyDrive/NLPProject/data/results/<condition>.jsonl`, so a disconnect resumes. When a condition finishes, a snapshot is written next to it as `<condition>_YYYY-MM-DD_HH-MM-SSZ.json`.

In [ ]:
BRANCH = "users/brianchang/testing"

# Check the GPU. Dependencies are installed from pyproject.toml after the clone.
!nvidia-smi -L

In [ ]:
import os
import subprocess
from pathlib import Path
from google.colab import drive, userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GH_TOKEN"] = userdata.get("GITHUB_TOKEN")

drive.mount("/content/drive")

dest = "/content/NLP-Project"
repo = "https://github.com/brianchang12/NLP-Project.git"

!gh auth setup-git

# Clone or update the repository
if not Path(dest).exists():
    !git clone {repo} {dest}
else:
    !git -C {repo} pull --ff-only

!git checkout {BRANCH}
%cd /content/NLP-Project
# Install the dependencies declared in pyproject.toml.
!pip install -q -U .

In [ ]:
condition = "baseline"  # @param ["baseline", "before_question", "beginning", "middle"]

In [ ]:
scripts = {
    "baseline": "nlp_project/src/nlp_project/scripts/eval_baseline.py",
    "before_question": "nlp_project/src/nlp_project/scripts/eval_before_question.py",
    "beginning": "nlp_project/src/nlp_project/scripts/eval_beginning.py",
    "middle": "nlp_project/src/nlp_project/scripts/eval_middle.py",
}
out_dir = "/content/drive/MyDrive/NLPProject/data/results"
script = scripts[condition]
%cd /content/NLP-Project
!python {script} --out-dir {out_dir}